# Time Series Forecasting of Sales Using Python

## Time Series Analysis and Forecasting

### Project Overview

Forecasting future values is a fundamental component of data-driven decision-making, enabling organizations to anticipate demand, optimize resource allocation, and improve strategic planning. Time series forecasting uses historical observations collected over time to identify underlying patterns such as trends, seasonality, and cyclical behavior, allowing future values to be predicted with greater confidence.

This project focuses on analyzing a historical time series dataset to uncover temporal patterns and develop forecasting models capable of predicting future observations. The project follows a structured workflow beginning with data understanding and quality assessment, followed by exploratory time series analysis, feature engineering, model development, and forecast evaluation.

---

## Project Objectives

The main objectives of this project are to:

- Understand the structure and characteristics of the time series dataset.
- Assess and validate the quality of the dataset before modeling.
- Explore temporal patterns, including trends, seasonality, and cyclical movements.
- Prepare the dataset for time series forecasting.
- Build and compare forecasting models.
- Evaluate forecasting performance using appropriate accuracy metrics.
- Generate future forecasts based on historical observations.
- Provide business insights and recommendations from the forecasting results.

---

## Project Structure

This notebook follows a structured time series forecasting workflow consisting of the following stages:

### 1. Data Understanding

The project begins by exploring the dataset to understand its structure and identify the variables required for forecasting. This stage includes:

- Loading the dataset
- Inspecting the dataset structure
- Reviewing data types
- Understanding the target variable
- Examining the time variable

### 2. Data Quality Assessment

Although the dataset requires minimal cleaning, a series of validation checks are performed to ensure the data is suitable for forecasting. These checks include:

- Missing value analysis
- Duplicate record detection
- Data type validation
- Date format verification
- Missing timestamp detection
- Data consistency checks

### 3. Exploratory Time Series Analysis (ETSA)

This stage focuses on understanding the behavior of the time series over time through visualization and statistical analysis. Activities include:

- Time series visualization
- Trend analysis
- Seasonality analysis
- Distribution analysis
- Rolling statistics
- Time-based feature exploration

### 4. Time Series Preparation

The dataset is prepared for forecasting by performing the necessary preprocessing steps, including:

- Setting the datetime index
- Checking stationarity
- Applying transformations where necessary
- Time series decomposition
- Autocorrelation analysis

### 5. Forecasting Model Development

Several forecasting techniques are developed and compared to determine the most suitable model for predicting future observations. This stage includes:

- Model training
- Forecast generation
- Performance evaluation
- Model comparison.

### 6. Forecast Visualization and Business Insights

The final stage presents the forecasting results and interprets their business implications. This includes:

- Visualizing historical and forecasted values
- Evaluating forecast accuracy
- Identifying future trends
- Providing actionable business recommendations

---

## Dataset Description

The dataset contains historical retail sales records collected over time and is designed to support time series forecasting and business planning. Each observation represents the sales performance of a retail business on a specific date, along with the corresponding inventory levels and product prices.

The dataset consists of four variables:

- Date – The timestamp for each observation.
- Sales – The total sales recorded during each time period. This serves as the target variable for forecasting.
- Inventory – The available stock level corresponding to each observation.
- Price – The selling price of the product during each time period.

These variables provide insight into historical sales performance and the business factors that may influence demand over time. By analyzing these historical observations, the project aims to identify temporal patterns and generate accurate forecasts of future sales.

---

## Problem Statement

Retail businesses operate in highly dynamic markets where customer demand fluctuates due to changing purchasing behavior, pricing strategies, inventory availability, seasonal trends, and broader market conditions. Without reliable forecasts, businesses may struggle to maintain optimal inventory levels, resulting in stock shortages that lead to lost sales or excess inventory that increases storage costs and ties up working capital. Accurate sales forecasting enables retailers to make informed decisions regarding inventory replenishment, pricing strategies, staffing, procurement, and overall business planning. However, producing reliable forecasts requires understanding historical sales patterns, identifying trends and seasonal behavior, and selecting forecasting models capable of capturing these temporal dynamics.

The resulting forecasts can support more effective inventory management, improve operational planning, and help the business make proactive, data-driven decisions in response to future demand.

---

## Expected Outcome

By the end of this project, the dataset will have been thoroughly assessed and analyzed, resulting in:

- A validated and well-understood time series dataset
- Clear identification of trends and seasonal patterns
- Comprehensive exploratory time series analysis
- Well-performing forecasting models
- Accurate future predictions
- Actionable business insights and recommendations based on the forecast results

The forecasting models developed in this project will demonstrate how historical time series data can be transformed into valuable predictive insights that support informed business decision-making.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Data Loading and Initial Inspection
Before cleaning the data, we inspect its structure to understand the available variables, identify data types, and detect potential issues such as missing values or incorrect formats.

In [ ]:
df = pd.read_csv("time_series_data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

### Note

The dataset contains **937 observations and 4 variables**, representing historical retail data that will be used to forecast future sales. The dataset includes a **date variable** and three business-related variables: **`venda` (sales), `estoque` (inventory), and `preco` (price)**.

An initial inspection was carried out to understand the structure, data types, statistical characteristics, and overall quality of the dataset. The dataset shape confirmed that there were 937 records, while the `info()` output was used to examine the data types and completeness of each variable. Descriptive statistics were also reviewed to understand the distribution and range of the numerical variables.

The missing-value check showed **0 null values** across all variables, meaning no missing-data treatment was required at this stage. Overall, the initial inspection indicated that the dataset was complete and suitable for further time-series preparation and analysis.


# Data Cleaning & Time Series Preparation

In [ ]:
df.drop_duplicates(inplace=True)
df.rename(columns={'data': 'date'}, inplace=True)
df.rename(columns={'venda': 'sales'}, inplace=True)
df.rename(columns={'estoque': 'inventory'}, inplace=True)
df.rename(columns={'preco': 'price'}, inplace=True)

In [ ]:
numeric_cols = df.select_dtypes(include='number')

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')
df = df.sort_values('date')
df = df.set_index('date')

full_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='D'
)

missing_dates = full_range.difference(df.index)

print(f"Expected dates: {len(full_range)}")
print(f"Actual dates: {len(df)}")
print(f"Missing dates: {len(missing_dates)}")

### Note

Before beginning the time-series analysis, data cleaning and preparation were performed to ensure that the dataset was consistent and suitable for modelling.

First, **duplicate records were checked and removed** to prevent repeated observations from influencing the analysis. The original Portuguese column names were then renamed to more descriptive English names:

* `venda` → `sales`
* `estoque` → `inventory`
* `preco` → `price`

This makes the dataset easier to interpret and allows the variables to be referenced consistently throughout the analysis and modelling stages.

Outlier analysis was also performed on the numerical variables. Some observations were identified as potential outliers in **sales (34), inventory (80), and price (37)**. However, these observations were not removed. In a retail forecasting context, unusually high or low values may represent genuine business events, such as periods of unusually high demand, changes in pricing, or inventory fluctuations. Removing them without evidence that they are data-entry errors could remove important information from the time series. Therefore, the identified outliers were retained at this stage, and their potential impact was considered during the subsequent exploratory analysis and modelling.

Also, the date column was converted to a datetime format, and the observations were sorted chronologically. The date was then set as the DataFrame index, which is important for time-series analysis because it allows observations to be ordered and analyzed according to their point in time.

The dataset was also checked for its underlying time frequency. A complete daily date range was generated between the earliest and latest observations and compared with the existing dates. This was done to identify any gaps in the time series that could affect time-based analysis, rolling calculations, and forecasting. The frequency check revealed that the dataset does not form a perfectly continuous daily sequence. This is important to account for when interpreting the time series and creating lag and rolling features later in the analysis.

# Exploratory Time Series Analysis

### A. Understanding Sales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Daily sales over time
axes[0].plot(df.index, df['sales'])
axes[0].set_title('Daily Sales Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales')

# 2. Sales distribution
axes[1].hist(df['sales'], bins=50)
axes[1].set_title('Sales Distribution')
axes[1].set_xlabel('Sales')
axes[1].set_ylabel('Count')

# 3. Sales with rolling averages
df['sales_7day'] = df['sales'].rolling(7).mean()
df['sales_30day'] = df['sales'].rolling(30).mean()

axes[2].plot(df.index, df['sales'], alpha=0.3)
axes[2].plot(df.index, df['sales_7day'], linewidth=2)
axes[2].plot(df.index, df['sales_30day'], linewidth=3)
axes[2].set_title('Daily Sales with Rolling Averages')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Sales')
axes[2].legend(['Daily Sales', '7-Day Average', '30-Day Average'])

plt.tight_layout()
plt.show()

### Note

Before examining the influence of other variables such as price and inventory, the sales variable was explored independently to understand its overall behavior over time.

Three complementary views were used. The first shows the **daily sales series**, allowing trends, fluctuations, and unusual periods to be observed. The second examines the **distribution of sales values**, providing insight into the typical range of sales and whether the data is concentrated around certain values or contains extreme observations. Finally, **7-day and 30-day rolling averages** were calculated to smooth out short-term fluctuations and make broader patterns in sales easier to identify.

Together, these visualizations provide an initial understanding of the level, variability, distribution, and underlying movement of sales before moving on to more detailed time-series analysis.


### B. Identify Seasonality

In [ ]:
df['day_name'] = df.index.day_name()
weekday_sales = df.groupby('day_name')['sales'].mean()
days = ['Monday','Tuesday','Wednesday','Thursday',
        'Friday','Saturday','Sunday']
weekday_sales = weekday_sales.reindex(days)


df['month_name'] = df.index.month_name()
monthly_sales = df.groupby('month_name')['sales'].mean()
months = ['January', 'February', 'March', 'April', 'May', 'June',
          'July', 'August', 'September', 'October', 'November', 'December']
monthly_sales = monthly_sales.reindex(months)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 5))

# 1. Weekly sales over time
axes[0].bar(weekday_sales.index, weekday_sales.values)
axes[0].set_title('Average Sales by Day of Week')
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Average Sales')

# 2. Monthly sales over time
axes[1].plot(monthly_sales.index, monthly_sales.values)
axes[1].set_title('Average Sales by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Sales')

plt.tight_layout()
plt.show()

### Note

After examining the overall sales trend, the next step was to investigate whether sales followed recurring patterns over time. Identifying seasonality is important in time-series forecasting because repeated patterns can provide useful information about future sales.

Two seasonal patterns were examined: **day-of-week effects** and **monthly effects**. Average sales were calculated for each day of the week and each month to determine whether sales consistently varied across these time periods.

The day-of-week analysis showed that average sales were highest on **Saturday**, while the lowest average sales occurred on **Tuesday**. This suggests that customer purchasing behavior may vary depending on the day of the week.

The monthly analysis showed that average sales were highest in **June** and lowest in **August**. This indicates that there may also be recurring patterns across different months of the year.

These patterns suggest that calendar-based features may provide useful information for forecasting and will therefore be considered during the feature engineering stage.


### C. Sales Patterns

In [ ]:
df['price_round'] = df['price'].round(1)
price_sales = df.groupby('price_round')['sales'].mean().reset_index()


In [ ]:
fig, axes =  plt.subplots(1, 2, figsize=(20, 5))

# 1. Average sales by price
axes[0].bar(price_sales['price_round'], price_sales['sales'])
axes[0].set_title('Average Sales by Price')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Average Sales')

# 2. Average sales by inventory
axes[1].scatter(df['inventory'], df['sales'], alpha=0.6)
axes[0].set_title('Sales vs Inventory')
axes[0].set_xlabel('Inventory')
axes[0].set_ylabel('Sales')

plt.tight_layout()
plt.show()

In [ ]:
df_weekly = df[['sales', 'inventory']].resample('W').agg({
    'sales': 'sum',
    'inventory': 'mean'
})

scaler = MinMaxScaler()

df_weekly[['sales_norm', 'inventory_norm']] = scaler.fit_transform(
    df_weekly[['sales', 'inventory']]
)

plt.figure(figsize=(10, 5))

plt.plot(
    df_weekly.index,
    df_weekly['sales_norm'],
    label='Weekly Sales'
)

plt.plot(
    df_weekly.index,
    df_weekly['inventory_norm'],
    label='Weekly Inventory'
)

plt.title('Weekly Sales and Inventory Over Time')
plt.xlabel('Week')
plt.ylabel('Normalized Value')

plt.legend()

plt.show()

### Note

After examining the time-based patterns in sales, the next step was to investigate whether the other variables in the dataset could provide additional information about sales behavior. In particular, **price and inventory** were examined because both may influence or be associated with customer demand and sales performance.

Firstly, prices were grouped by rounded price values, and the average sales associated with each price level were calculated. This provides an initial view of how sales behavior changes across different price points.
The visualization shows that average sales first increased as price increased the plummeted after price crossed 2.2. The highest average sales occurred around a price of **2.0**, while the lowest average sales occurred around **3.4**. This analysis provides an initial indication of whether price may contain useful information for predicting sales. However, the relationship observed in this visualization should not be interpreted as causation, since other factors such as seasonality, inventory levels, and changes in customer demand may also influence sales.

The second visualization shows the relationship between inventory and sales using a scatter plot.  The correlation between inventory and sales was approximately **0.154**, indicating a **weak positive relationship** between the two variables.

These observations provide an initial understanding of the potential explanatory variables that can be incorporated into the forecasting models. Later in the project, their usefulness will be tested by comparing models based only on historical sales with models that also incorporate price and inventory information.


### D. Correlation Analysis

In [ ]:
corr_matrix = df[['sales','price','inventory']].corr()

corr_matrix

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,5))

sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt='.2f')

plt.title('Correlation Matrix')

plt.show()

### Note

To complement the individual visualizations, a correlation matrix was used to examine the strength and direction of the linear relationships between **sales, price, and inventory**.

The results show a **very weak positive correlation between sales and inventory (0.154)**, suggesting that higher inventory levels are only weakly associated with higher sales. The correlation between **sales and price (0.095)** is also very weak and positive, indicating little linear relationship between the two variables in this dataset.

There is almost no linear relationship between **price and inventory (-0.033)**, with the value being very close to zero.

Overall, the correlation analysis suggests that neither price nor inventory has a strong linear relationship with sales on its own. However, weak correlation does not necessarily mean that these variables have no predictive value. They may still contribute to forecasting when combined with historical sales patterns, time-based features, and non-linear modelling techniques.

These findings support the project's central modelling question: whether incorporating price and inventory can improve sales forecasts beyond what can be achieved using historical sales patterns alone.


# Feature Engineering

In [ ]:
# Time-based features
df['year'] = df.index.year
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['week'] = df.index.isocalendar().week
df['day_of_week'] = df.index.dayofweek

# Lag features
df['sales_lag_1'] = df['sales'].shift(1)
df['sales_lag_7'] = df['sales'].shift(7)
df['sales_lag_14'] = df['sales'].shift(14)
df['sales_lag_28'] = df['sales'].shift(28)

# Rolling averages using previous observations only
df['sales_rolling_7'] = df['sales'].shift(1).rolling(7).mean()
df['sales_rolling_14'] = df['sales'].shift(1).rolling(14).mean()
df['sales_rolling_30'] = df['sales'].shift(1).rolling(30).mean()

# Changes in price and inventory
df['price_change'] = df['price'].diff()
df['inventory_change'] = df['inventory'].diff()

# Remove temporary exploratory columns
col_del = ['price_round', 'day_name', 'month_name']
df = df.drop(columns=col_del)

# Check and remove missing values created by feature engineering
print(df.isnull().sum())

df = df.dropna()

print(f"Dataset shape after feature engineering: {df.shape}")

### Note

To prepare the dataset for forecasting models, additional features were created to capture **time-based patterns, historical sales behavior, and changes in price and inventory**. The goal was to provide the models with information that may help explain and predict future sales.

#### 1. Time-Based Features

Calendar features were extracted from the datetime index:

* **Year** – captures longer-term changes in sales across years.
* **Month** – captures monthly and seasonal patterns.
* **Quarter** – captures broader seasonal patterns across the year.
* **Week** – captures changes across different weeks of the year.
* **Day of Week** – captures recurring weekly purchasing patterns.

These features allow the models to learn whether sales behave differently depending on the time of year or day of the week.

#### 2. Lag Features

Lag features were created using previous sales values:

* **1-day lag** – sales from the previous day.
* **7-day lag** – sales from the previous week.
* **14-day lag** – sales from two weeks earlier.
* **28-day lag** – sales from four weeks earlier.

Lag features are particularly useful in time-series forecasting because recent historical sales can provide information about future sales. Using multiple lag periods also allows the models to capture both short-term and recurring patterns.

#### 3. Rolling Average Features

Rolling averages were calculated over 7, 14, and 30 days to capture recent sales trends while reducing the effect of individual daily fluctuations.

The rolling averages were calculated using the **previous observations only** by shifting the sales series by one day before applying the rolling window. This prevents information from the current day from being used to predict that same day's sales and helps avoid data leakage.

#### 4. Price and Inventory Change

Two additional features were created to capture changes in the business environment:

* **Price Change** – the difference between the current price and the previous day's price.
* **Inventory Change** – the difference between the current inventory level and the previous day's inventory.

Rather than only providing the absolute price and inventory values, these features allow the models to identify whether recent changes in price or inventory are associated with changes in sales.

#### 5. Removing Unnecessary Columns

Temporary columns created during exploratory analysis, such as `price_round`, `day_name`, and `month_name`, were removed because their numerical or categorical counterparts were already represented by the engineered features.

The newly created lag and rolling features naturally produced missing values at the beginning of the dataset because there were not enough previous observations available to calculate them. These missing values were identified and the affected rows were removed before modelling.

After feature engineering and cleaning, the resulting dataset contained the variables required for the forecasting models while preserving the chronological structure of the time series.


# Train-Test Split

In [ ]:
# Check for gaps greater than one day
df.index.to_series().diff().loc[
    df.index.to_series().diff() > pd.Timedelta(days=1)
]

In [ ]:
train = df[df.index.year < 2016]
test = df[df.index.year == 2016]

print(train.shape)
print(test.shape)
print(train.index.min(), train.index.max())
print(test.index.min(), test.index.max())

### Note

For time-series forecasting, the data must be split chronologically rather than randomly. A random train/test split could allow observations from the future to enter the training data, resulting in **data leakage** and an unrealistic estimate of model performance.

The dataset was therefore divided based on the year:

* **Training set:** observations from 2014–2015
* **Test set:** observations from 2016

The training set represents the historical information that would have been available when making a forecast, while the test set represents a future period that the models have not seen during training.

Before performing the split, the time index was also checked for gaps greater than one day. This confirms whether there are breaks in the daily sequence that should be considered when interpreting lag-based features and forecasting results.

The chronological split ensures that model performance is evaluated on genuinely future observations, making the evaluation more representative of how the model would perform in a real-world forecasting scenario.


# Forecasting

With the dataset prepared and the train/test period defined, the next step was to build forecasting models.

Rather than immediately moving to a complex model, the analysis follows a progressive approach. Three approaches were compared:

1. **Naive Forecast** – establishes a simple baseline using historical sales.
2. **Univariate Model** – uses only historical sales information and engineered sales-based features.
3. **Multivariate Model** – combines historical sales information with price and inventory variables.

This comparison is important because it helps determine not only which model performs best, but also **whether additional business variables actually improve forecasting performance**.

The models were evaluated on the same 2016 test period using **Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE)**.


# Baseline Forecast

In [ ]:
baseline_pred = df['sales'].shift(1).loc[test.index]
y_test = test['sales']
baseline = pd.DataFrame({
    'actual': y_test,
    'predicted': baseline_pred
})

baseline.head(5)

In [ ]:
(y_test == 0).sum()

In [ ]:
mae_baseline = mean_absolute_error(y_test, baseline_pred)

rmse_baseline = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

non_zero = y_test != 0

mape_baseline = np.mean(
    np.abs(
        (y_test[non_zero] - baseline_pred[non_zero])
        / y_test[non_zero]
    )
) * 100

print(f"MAE: {mae_baseline:.2f}")
print(f"RMSE: {rmse_baseline:.2f}")
print(f"MAPE: {mape_baseline:.2f}%")

### Note

The **Naive Forecast** was used as the baseline model. It assumes that the best prediction for the next period is simply the most recent observed sales value.

In other words, the forecast for each day is based on the sales value from the previous day. Although this approach is very simple, it provides an important benchmark. A more complex machine-learning model should only be considered useful if it can consistently outperform this baseline on unseen data.
The Naive Forecast therefore establishes the minimum level of performance that the more advanced models need to beat.


# Univariate Forecast

In [ ]:
univariate_features = [
    'sales_lag_1',
    'sales_lag_7',
    'sales_lag_14',
    'sales_lag_28',
    'sales_rolling_7',
    'sales_rolling_14',
    'sales_rolling_30'
]

X_train_uni = train[univariate_features]
y_train_uni = train['sales']

X_test_uni = test[univariate_features]
y_test_uni = test['sales']

## Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model = rf.fit(X_train_uni, y_train_uni)
pred_y = model.predict(X_test_uni)


mae_uni = mean_absolute_error(y_test_uni, pred_y)
rmse_uni = np.sqrt(mean_squared_error(y_test_uni, pred_y))

non_zero = y_test_uni != 0

mape_uni = np.mean(
    np.abs((y_test_uni[non_zero] - pred_y[non_zero])/ y_test_uni[non_zero])) * 100

print(f"MAE: {mae_uni:.2f}")
print(f"RMSE: {rmse_uni:.2f}")
print(f"MAPE: {mape_uni:.2f}%")

## XGBoost

In [ ]:
params = {
    'objective': 'reg:squarederror',
    'max_depth': 4,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1
}
xgb_uni = XGBRegressor(
    **params,
    random_state=42
)

model = xgb_uni.fit(X_train_uni, y_train_uni)

pred_y = model.predict(X_test_uni)
mae_xgb = mean_absolute_error(y_test_uni, pred_y)
rmse_xgb = np.sqrt(mean_squared_error(y_test_uni, pred_y))

non_zero = y_test_uni != 0

mape_xgb = np.mean(
    np.abs((y_test_uni[non_zero] - pred_y[non_zero])/ y_test_uni[non_zero])) * 100

print(f"MAE: {mae_xgb:.2f}")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"MAPE: {mape_xgb:.2f}%")

### Note

The next approach was a **univariate forecasting**, where the models(Random Forest and XGBoost) were given information derived only from the historical sales series.

The model used lagged sales and rolling sales features to capture patterns such as recent sales activity, weekly behavior, and longer-term movement.

The main features included:

* Previous-day sales
* Previous-week sales
* Sales from two weeks earlier
* Sales from four weeks earlier
* 7-day rolling average
* 14-day rolling average
* 30-day rolling average
* Calendar-based features such as month, quarter, week, and day of week

This approach tests whether **historical sales patterns alone contain enough information to produce useful forecasts**.

Because no price or inventory information is provided to the model, its performance represents how much predictive information can be extracted from the sales history itself.


# Multivariate Forecast

In [ ]:
multivariate_features = [
    'sales_lag_1',
    'sales_lag_7',
    'sales_lag_14',
    'sales_lag_28',
    'sales_rolling_7',
    'sales_rolling_14',
    'sales_rolling_30',
    'price',
    'inventory',
    'price_change',
    'inventory_change'
]

X_train_multi = train[multivariate_features]
y_train_multi = train['sales']

X_test_multi = test[multivariate_features]
y_test_multi = test['sales']

## Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model = rf.fit(X_train_multi, y_train_multi)
pred_y_multi = model.predict(X_test_multi)


mae_multi = mean_absolute_error(y_test_multi, pred_y_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test_multi, pred_y_multi))

non_zero = y_test_multi != 0

mape_multi = np.mean(
    np.abs((y_test_multi[non_zero] - pred_y_multi[non_zero])/ y_test_multi[non_zero])) * 100

print(f"MAE: {mae_multi:.2f}")
print(f"RMSE: {rmse_multi:.2f}")
print(f"MAPE: {mape_multi:.2f}%")

## XGBoost

In [ ]:
params = {
    'objective': 'reg:squarederror',
    'max_depth': 4,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1
}
xgb_uni = XGBRegressor(
    **params,
    random_state=42
)

model = xgb_uni.fit(X_train_multi, y_train_multi)

pred_y = model.predict(X_test_multi)
mae_xgb = mean_absolute_error(y_test_multi, pred_y)
rmse_xgb = np.sqrt(mean_squared_error(y_test_multi, pred_y))

non_zero = y_test_multi != 0

mape_xgb = np.mean(
    np.abs((y_test_multi[non_zero] - pred_y[non_zero])/ y_test_multi[non_zero])) * 100

print(f"MAE: {mae_xgb:.2f}")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"MAPE: {mape_xgb:.2f}%")

### Note

The third approach extended the univariate model by incorporating additional business variables: **price and inventory**.

The multivariate model therefore had access to both:

* **Historical sales information**, including lag and rolling features
* **Business variables**, including price, inventory, price changes, and inventory changes

The purpose of this model was to determine whether information beyond the sales history could improve forecasting accuracy.

This comparison is particularly relevant given the earlier exploratory analysis. The correlation analysis showed only weak linear relationships between sales and the other variables, with correlations of approximately **0.095 between sales and price** and **0.154 between sales and inventory**.

However, weak correlation does not necessarily mean that these variables have no predictive value. A machine-learning model can potentially identify **non-linear relationships or interactions** that are not captured by simple correlation coefficients.

The multivariate model therefore provides a practical test of whether price and inventory add predictive value when combined with historical sales patterns.


# Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Naive',
        'Random Forest',
        'Random Forest',
        'XGBoost',
        'XGBoost'
    ],
    'Type': [
        'Baseline',
        'Univariate',
        'Multivariate',
        'Univariate',
        'Multivariate'
    ],
    'RMSE': [
        74.16,
        67.70,
        40.41,
        75.41,
        44.14
    ],
    'MAE': [
        52.81,
        47.54,
        20.53,
        53.81,
        25.22
    ],
    'MAPE': [
        114.26,
        133.34,
        55.33,
        152.51,
        72.97
    ]
})

results

In [ ]:
models = [
    'Naive',
    'RF Univariate',
    'RF Multivariate',
    'XGB Univariate',
    'XGB Multivariate'
]

rmse_values = results['RMSE']
mae_values = results['MAE']
mape_values = results['MAPE']

pairs = [
    ('RMSE', rmse_values),
    ('MAE', mae_values),
    ('MAPE', mape_values)
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (metric, values) in zip(axes, pairs):

    ax.bar(models, values)

    ax.set_title(f'Model Comparison - {metric}')
    ax.set_xlabel('Model')
    ax.set_ylabel(metric)

    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Note

The three forecasting approaches were evaluated on the same unseen test period using MAE, RMSE, and MAPE.

The **Naive Forecast** achieved an MAE of **52.81**, an RMSE of **74.16**, and a MAPE of **114.26%**. This established the baseline performance.

The **Univariate Random Forest** produced an MAE of **53.81**, an RMSE of **75.41**, and a MAPE of **152.51%**. Although the model was considerably more sophisticated than the Naive Forecast, it did not outperform the baseline on these metrics.

The **Multivariate Random Forest**, however, produced substantially better results, with an MAE of **25.22**, an RMSE of **44.14**, and a MAPE of **72.97%**.

This represents a significant improvement over both the Naive and Univariate approaches. In particular, the reduction in MAE from **53.81 to 25.22** suggests that incorporating price and inventory information provided additional predictive information that was not captured by historical sales patterns alone.

### What the Comparison Tells Us

The comparison provides an important finding for the forecasting problem. **Historical sales information alone was not sufficient to outperform the simple Naive baseline in this experiment.** However, adding price and inventory variables resulted in a substantial improvement in predictive performance.

This suggests that sales are influenced by information beyond their own historical pattern, even though price and inventory showed only weak linear correlations with sales during exploratory analysis.

The results therefore support the use of the **multivariate approach** as the foundation for further model improvement. The next step is to investigate whether the model's performance can be improved further through **hyperparameter tuning** before using the final model to generate forecasts for future sales.



## Random Forest Hpyerparameter Tuning

In [ ]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_multi, y_train_multi)

In [ ]:
random_search.best_params_

In [ ]:
random_search.best_score_

### Note

After comparing the Naive, Univariate, and Multivariate approaches, the **Multivariate Random Forest** was selected for further improvement because it produced the strongest forecasting performance.

Rather than using the initial Random Forest configuration, hyperparameter tuning was performed to search for a combination of parameters that could improve the model's predictive performance.

The best tuning score was **-41.5**. Because the tuning process used a scoring metric where lower error is preferred and the score was expressed as a negative value, the result corresponds to an estimated **MAE of approximately 41.5 during cross-validation**.
The selected configuration uses **500 trees**, allowing the Random Forest to average predictions across a large number of decision trees. At the same time, the maximum tree depth was limited to **5**, which helps control model complexity and reduce the risk of overfitting. A minimum of **2 observations per leaf** also provides some regularization while allowing the trees to capture useful patterns.

The tuning results provided a more controlled Random Forest configuration to use for the final evaluation on the held-out 2016 test set.


## 21-day Rolling Backtest Comparison
### Recursive RF vs Direct RF

In [ ]:
def direct_forecast(model_dict, history_df, features):

    # Use the most recent known feature row at the forecast origin
    X_future = history_df.iloc[[-1]][features]

    predictions = []

    for horizon in range(1, 22):

        model = model_dict[horizon]

        prediction = model.predict(X_future)[0]

        predictions.append(prediction)

    return np.array(predictions)


def recursive_forecast(model, history_df, future_dates, features):

    # Create a working dataframe containing history + future dates
    forecast_df = history_df.copy()

    forecast_df = forecast_df.reindex(
        forecast_df.index.union(future_dates)
    )


    last_price = history_df['price'].iloc[-1]
    last_inventory = history_df['inventory'].iloc[-1]

    forecast_df.loc[future_dates, 'price'] = last_price
    forecast_df.loc[future_dates, 'inventory'] = last_inventory

    forecast_df.loc[future_dates, 'price_change'] = 0
    forecast_df.loc[future_dates, 'inventory_change'] = 0

    predictions = []

    
    for date in future_dates:

        # Lag features
        forecast_df.loc[date, 'sales_lag_1'] = (
            forecast_df['sales'].shift(1).loc[date]
        )

        forecast_df.loc[date, 'sales_lag_7'] = (
            forecast_df['sales'].shift(7).loc[date]
        )

        forecast_df.loc[date, 'sales_lag_14'] = (
            forecast_df['sales'].shift(14).loc[date]
        )

        forecast_df.loc[date, 'sales_lag_28'] = (
            forecast_df['sales'].shift(28).loc[date]
        )

        # Rolling features
        forecast_df.loc[date, 'sales_rolling_7'] = (
            forecast_df['sales']
            .shift(1)
            .rolling(7)
            .mean()
            .loc[date]
        )

        forecast_df.loc[date, 'sales_rolling_14'] = (
            forecast_df['sales']
            .shift(1)
            .rolling(14)
            .mean()
            .loc[date]
        )

        forecast_df.loc[date, 'sales_rolling_30'] = (
            forecast_df['sales']
            .shift(1)
            .rolling(30)
            .mean()
            .loc[date]
        )

        # Create feature row
        X_future = forecast_df.loc[[date], features]

        # Predict
        prediction = model.predict(X_future)[0]

        # Store prediction so future lags/rolling features can use it
        forecast_df.loc[date, 'sales'] = prediction

        predictions.append(prediction)

    return np.array(predictions)

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="`sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel`"
)

In [ ]:
final_features = [
    'sales_lag_1',
    'sales_lag_7',
    'sales_lag_14',
    'sales_lag_28',
    'sales_rolling_7',
    'sales_rolling_14',
    'sales_rolling_30',
    'price',
    'inventory',
    'price_change',
    'inventory_change'
]

cutoff_dates = [
    pd.Timestamp('2015-12-31'),
    pd.Timestamp('2016-01-31'),
    pd.Timestamp('2016-02-29'),
    pd.Timestamp('2016-03-31'),
    pd.Timestamp('2016-04-30'),
    pd.Timestamp('2016-05-31')
]

comparison_results = []

for cutoff in cutoff_dates:

    print(f"\nRunning cutoff: {cutoff.date()}")

    history = df.loc[:cutoff].copy()

    # Next 21 calendar days
    future_dates = pd.date_range(
        start=cutoff + pd.Timedelta(days=1),
        periods=21,
        freq='D'
    )

    # Make sure all actual future values exist
    if not future_dates.isin(df.index).all():

        print(
            f"Skipping {cutoff.date()} - "
            f"missing future dates"
        )

        continue

    # 2. TRAIN DATA
    
    training_data = history.dropna(
        subset=final_features + ['sales']
    )

    X_train = training_data[final_features]
    y_train = training_data['sales']

   
    # 3. RECURSIVE RANDOM FOREST
    

    recursive_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=5,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=1
    )

    recursive_model.fit(
        X_train,
        y_train
    )

    # 4. RECURSIVE 21-DAY FORECAST

    recursive_predictions = recursive_forecast(
        recursive_model,
        history,
        future_dates,
        final_features
    )

    # 5. TRAIN 21 DIRECT MODELS

    direct_history = history.copy()

    direct_models_bt = {}

    for horizon in range(1, 22):

        target = f'target_{horizon}'

        # Create horizon-specific target
        direct_history[target] = (
            direct_history['sales'].shift(-horizon)
        )

        # Only use rows with complete features
        # and a known future target
        training_data_direct = direct_history.dropna(
            subset=final_features + [target]
        )

        X_direct = training_data_direct[final_features]
        y_direct = training_data_direct[target]

        model = RandomForestRegressor(
            n_estimators=500,
            max_depth=5,
            min_samples_split=2,
            min_samples_leaf=2,
            max_features=1.0,
            random_state=42,
            n_jobs=1
        )

        model.fit(
            X_direct,
            y_direct
        )

        direct_models_bt[horizon] = model

    # 6. DIRECT 21-DAY FORECAST

    direct_predictions = direct_forecast(
        direct_models_bt,
        history,
        final_features
    )

    # 7. ACTUAL VALUES

    actual = df.loc[
        future_dates,
        'sales'
    ].values

    # 8. RECURSIVE METRICS

    recursive_mae = mean_absolute_error(
        actual,
        recursive_predictions
    )

    recursive_rmse = np.sqrt(
        mean_squared_error(
            actual,
            recursive_predictions
        )
    )

    non_zero = actual != 0

    recursive_mape = np.mean(
        np.abs(
            (
                actual[non_zero]
                - recursive_predictions[non_zero]
            )
            / actual[non_zero]
        )
    ) * 100

    # 9. DIRECT METRICS

    direct_mae = mean_absolute_error(
        actual,
        direct_predictions
    )

    direct_rmse = np.sqrt(
        mean_squared_error(
            actual,
            direct_predictions
        )
    )

    direct_mape = np.mean(
        np.abs(
            (
                actual[non_zero]
                - direct_predictions[non_zero]
            )
            / actual[non_zero]
        )
    ) * 100

    # 10. SAVE RESULTS

    comparison_results.append({

        'cutoff_date': cutoff,

        'Recursive_MAE': recursive_mae,
        'Direct_MAE': direct_mae,

        'Recursive_RMSE': recursive_rmse,
        'Direct_RMSE': direct_rmse,

        'Recursive_MAPE': recursive_mape,
        'Direct_MAPE': direct_mape
    })


In [ ]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df

In [ ]:
overall_comparison = pd.DataFrame({
    'Model': [
        'Recursive RF',
        'Direct RF'
    ],

    'MAE': [
        comparison_df['Recursive_MAE'].mean(),
        comparison_df['Direct_MAE'].mean()
    ],

    'RMSE': [
        comparison_df['Recursive_RMSE'].mean(),
        comparison_df['Direct_RMSE'].mean()
    ],

    'MAPE': [
        comparison_df['Recursive_MAPE'].mean(),
        comparison_df['Direct_MAPE'].mean()
    ]
})

overall_comparison

### Forecast Strategy Evaluation

A rolling-origin backtest was used to compare recursive and direct
21-day forecasting strategies across six historical cutoff dates.

Average performance:

Recursive:
- MAE: 83.26
- RMSE: 104.04

Direct:
- MAE: 65.30
- RMSE: 80.30

The direct approach produced lower average MAE and RMSE and
performed better in most of the evaluation windows, although
recursive forecasting performed better in some individual periods.

MAE was used as the primary selection metric, with RMSE as a
secondary metric. MAPE was treated as supplementary because
low/zero sales values can make percentage-based errors unstable.

Based on the rolling-origin evaluation, the direct multi-step
approach was selected for the final 21-day forecast.

# FINAL DIRECT 21-DAY FORECAST

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="`sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel`"
)

In [ ]:
final_features = [
    'sales_lag_1',
    'sales_lag_7',
    'sales_lag_14',
    'sales_lag_28',
    'sales_rolling_7',
    'sales_rolling_14',
    'sales_rolling_30',
    'price',
    'inventory',
    'price_change',
    'inventory_change'
]

# 1. Prepare training data

history = df.copy()

training_data = history.dropna(
    subset=final_features + ['sales']
)

X_train = training_data[final_features]

# 2. Train 21 direct models

direct_models = {}

for horizon in range(1, 22):

    target = history['sales'].shift(-horizon)

    valid = history[final_features].notna().all(axis=1) & target.notna()

    X = history.loc[valid, final_features]
    y = target.loc[valid]

    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=5,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y)

    direct_models[horizon] = model

# 3. Create future dates

last_date = df.index.max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=21,
    freq='D'
)

# 4. Create final forecast

# Use the latest available feature row
latest_features = df[final_features].iloc[-1:].copy()

final_predictions = []

for horizon in range(1, 22):

    prediction = direct_models[horizon].predict(
        latest_features
    )[0]

    final_predictions.append(prediction)


forecast = pd.DataFrame({
    'date': future_dates,
    'predicted_sales': final_predictions
})

forecast['predicted_sales'] = forecast['predicted_sales'].round(2)

print("\nFinal 21-Day Forecast:")
display(forecast)

print("\nTotal forecasted sales:",
      round(forecast['predicted_sales'].sum(), 2))

print("Average daily forecast:",
      round(forecast['predicted_sales'].mean(), 2))

In [ ]:
forecast['predicted_sales'].describe()

### Final 21-Day Forecast

The selected direct multi-step Random Forest approach was used to forecast sales for the next 21 days. Instead of recursively feeding each prediction into the next forecast, 21 separate models were trained, with each model predicting a specific future horizon from 1 to 21 days ahead.

The final forecast has a mean expected daily sales volume of **203.93 units**, with a standard deviation of **43.66 units**, indicating moderate variation across the forecast period. Predicted daily sales range from **142.41 to 336.81 units**, while the median forecast is **199.63 units**.

The forecast therefore suggests an overall expected demand level of approximately **4,282.58 units over the 21-day period**. The variation between the minimum and maximum predicted values indicates that demand is not expected to remain constant throughout the forecast horizon.

These predictions should be interpreted as a baseline estimate of expected demand rather than an exact representation of future sales. Actual demand may differ due to changes in pricing, inventory availability, customer behavior, promotions, and other factors not captured by the model.


# Business Recommendations

The purpose of the forecasting analysis is not only to predict future sales, but also to translate the findings into practical business decisions. Based on the exploratory analysis, model comparison, forecasting strategy evaluation, and 21-day forecast, the following recommendations can be made.

### 1. Which Forecasting Approach Should the Retailer Use?

The **Direct Multi-Step Random Forest approach** should be used for the final 21-day forecasting task.

The tuned Random Forest achieved:

* **MAE:** 18.64
* **RMSE:** 38.86
* **MAPE:** 57.76%

For the 21-day forecasting problem, recursive and direct forecasting strategies were subsequently compared using rolling-origin backtesting. The direct approach produced a lower average:

* **MAE:** 65.30, compared with 83.26 for recursive forecasting
* **RMSE:** 80.30, compared with 104.04 for recursive forecasting

Although the recursive approach performed better in some individual forecasting windows, the direct approach generally produced lower errors across the evaluation periods. It was therefore selected for the final 21-day forecast.

The forecasting system should be used as a planning tool, while predictions should continue to be compared with actual sales and the models retrained as new data becomes available.

### 2. How Much Inventory Should the Retailer Plan For?

The final direct forecasting approach predicts **21 days of future sales**, with:

* **Average predicted daily sales:** 203.93 units
* **Minimum predicted sales:** 142.41 units
* **Maximum predicted sales:** 336.81 units
* **Standard deviation:** 43.66 units
* **Total predicted sales over 21 days:** approximately **4,283 units**

The retailer could therefore use approximately **4,283 units as the baseline demand forecast for the next three weeks**.

However, this should not necessarily be treated as the exact inventory quantity to hold. Actual inventory requirements depend on factors such as supplier lead time, replenishment frequency, safety stock, and the cost of stockouts.

Because predicted demand varies considerably across the forecast period, the retailer should plan for periods of higher expected demand rather than relying solely on the average of 203.93 units per day. An appropriate safety-stock buffer should also be considered to account for forecast uncertainty and unexpected increases in demand.

### 3. Is Price an Important Driver of Demand?

The analysis does **not provide strong evidence that price has a strong linear relationship with sales**.

The correlation between price and sales was approximately **0.095**, indicating a very weak linear relationship. This means that price alone does not explain much of the variation in sales in this dataset.

However, price should not be dismissed completely. The multivariate model performed substantially better than the univariate model, suggesting that price and other business variables can provide useful predictive information when considered alongside historical sales patterns.

The retailer should therefore continue monitoring price alongside sales rather than assuming that price changes alone will reliably predict changes in demand.

### 4. Is Inventory Constraining Sales?

The analysis provides **limited evidence that inventory is strongly constraining sales**.

The correlation between inventory and sales was approximately **0.154**, indicating only a weak positive relationship. This suggests that higher inventory levels are associated with somewhat higher sales, but the relationship is not strong enough to conclude that inventory is a major driver of sales.

Therefore, the retailer should avoid assuming that simply increasing inventory will automatically produce higher sales.

Instead, inventory decisions should be based on the sales forecast, historical demand patterns, replenishment requirements, and observed periods of unusually high demand.

### 5. Recommended Business Actions

Based on the findings, the retailer should:

1. **Use the direct multi-step Random Forest for short-term forecasting**, particularly for planning over the next few weeks.

2. **Plan inventory around forecasted demand**, using the approximately **4,283-unit three-week forecast** as a baseline and adding an appropriate safety-stock buffer.

3. **Monitor actual sales against forecasts** regularly. Large differences between predicted and actual sales can indicate changing customer behavior or conditions that the model has not captured.

4. **Track price and inventory alongside sales** rather than evaluating them independently. Although their individual correlations with sales are weak, they may still provide useful information when combined with historical sales patterns.

5. **Pay particular attention to recurring demand patterns**, especially differences across days of the week and months, when planning stock and promotions.

6. **Retrain the forecasting models as new sales data becomes available** so that the forecasting system can adapt to changing demand patterns.

7. **Use Power BI for ongoing monitoring and decision-making**, bringing together historical sales, forecasts, price, inventory, and key performance indicators in one dashboard.

### Overall Recommendation

The analysis suggests that **historical sales alone are not enough to produce the strongest forecasts in this dataset**. The substantial improvement achieved by the multivariate model indicates that incorporating additional business information can make short-term forecasting more useful.

For the 21-day forecasting task, the **direct multi-step approach provided better average MAE and RMSE in rolling backtesting** and was therefore selected for the final forecast. The resulting forecast estimates approximately **4,283 units of demand over the next 21 days**, with expected daily demand ranging from approximately **142 to 337 units**.

The retailer should use these predictions as **planning estimates rather than guaranteed outcomes**, combining them with inventory monitoring, price analysis, safety-stock considerations, and regular model updates to support short-term operational decisions.


In [ ]:
df.to_csv('sales_data.csv', index=True)

In [ ]:
forecast.to_csv('21_day_sales_forecast.csv', index=True)